<div class="align-center">

<a href="https://rocm.docs.amd.com/projects/ai-developer-hub/en/latest/index.html"><img src="https://raw.githubusercontent.com/ROCm/gpuaidev/main/docs/images/rocm_logo.png" alt="ROCm AI Developer Hub" width="150" style="display:inline-block; margin-right: 20px;"></a>
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115" style="display:inline-block;"></a>

</div>

---

# Pokémon LLM Agent — companion demo (step by step)

**Authors:** Yue Yuan ([yueyuan](https://github.com/yueyuan), yueyuan@amd.com), Bill He ([billishyahao](https://github.com/billishyahao), bill.he@amd.com)

Hands-on companion to `pokemon_llm_agent_unsloth_rocm_tutorial.ipynb`. **Run this notebook from the repo root** that contains `scripts/eval/showdown_agent_eval.py`.

### What you will do (step by step)

| Step | Topic |
|------|--------|
| 0–2 | ROCm env, install, GPU check |
| 3 | **Learn Showdown rules** — images + PokéAPI / Showdown interactive demo |
| 4 | **Compare checkpoints** — base vs published weights on the same prompt |
| 5 | **Dataset processing** — long replay log → short training row |
| 6–7 | LoRA + short SFT |
| 8 | **GRPO + reward design** — format + `|request|` legality |
| 9–10 | Inference + export |

**Data:** [`milkkarten/pokemon-showdown-replays-merged`](https://huggingface.co/datasets/milkkarten/pokemon-showdown-replays-merged) (streaming, disk-safe)

**Reference weights:** [`GoldenGrapeGentleman1/pokemon-showdown-agent-v6`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-v6)


## Step 0 — AMD ROCm environment
Set cache paths and ROCm-friendly defaults before heavy imports.

In [ ]:

import os
from pathlib import Path
import datasets

def pick(cands):
    for c in cands:
        p = Path(c)
        try:
            p.mkdir(parents=True, exist_ok=True)
            t = p / ".w"; t.write_text("ok"); t.unlink()
            return str(p)
        except Exception:
            continue
    return cands[-1]

cache = pick(["/shared-docker/.cache/huggingface", "/data/huggingface", "/tmp/pokemon-hf-cache"])
tmp = pick(["/shared-docker/.cache/tmp", "/data/tmp", "/tmp/pokemon-tmp"])
os.environ.setdefault("HIP_VISIBLE_DEVICES", "0")
os.environ.setdefault("ROCR_VISIBLE_DEVICES", os.environ["HIP_VISIBLE_DEVICES"])
os.environ["HF_HOME"] = cache
os.environ["HF_DATASETS_CACHE"] = f"{cache}/datasets"
os.environ["TMPDIR"] = tmp
os.environ.setdefault("TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL", "1")
os.environ.setdefault("PYTORCH_HIP_ALLOC_CONF", "expandable_segments:False")
os.environ.setdefault("UNSLOTH_SKIP_TORCHVISION_CHECK", "1")
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
datasets.config.HF_DATASETS_CACHE = os.environ["HF_DATASETS_CACHE"]
print(f"HF_HOME={os.environ['HF_HOME']}\nTMPDIR={os.environ['TMPDIR']}")


## Step 1 — Installation

In [ ]:

%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install --no-cache-dir --no-deps unsloth unsloth_zoo
    !pip install --no-cache-dir transformers accelerate peft trl datasets psutil sentencepiece protobuf tyro huggingface_hub hf_transfer einops


## Step 2 — Validate ROCm / GPU

In [ ]:

import torch
import unsloth
print(f"torch={torch.__version__} hip={getattr(torch.version, 'hip', None)}")
if not torch.cuda.is_available():
    raise RuntimeError("No ROCm GPU visible.")
print(f"gpu={torch.cuda.get_device_name(0)} bf16={torch.cuda.is_bf16_supported()}")


## Step 3 — Learn Pokémon Showdown rules (step by step)

Each turn the agent outputs **one line**: `move <Name>` or `switch <Species>` (optional `terastallize`).

| Concept | Battle UI | Protocol |
|---------|-----------|----------|
| Active / bench | sprites + party | `|switch|p2a: Pikachu` |
| Move | animation | `|move|p2a: Pikachu|Thunderbolt|p1a: Charizard` |
| Tera | crystal overlay | `|terastallize|p2a: Pikachu|Electric` |

**Next cell** shows species sprites (online CDN with offline fallback).  
**Following cell** runs the mini `move`/`switch` demo (PokéAPI with built-in fallback when Docker blocks outbound HTTP).


In [ ]:
from IPython.display import HTML, display

# Offline-first gallery (no network required)
CARDS = [
    ("Pikachu", "electric", "#F8D030", "⚡"),
    ("Charizard", "fire", "#F08030", "🔥"),
    ("Garchomp", "dragon", "#7038F8", "🐉"),
]
html = '<div style="display:flex;gap:10px;flex-wrap:wrap">'
for name, typ, color, emoji in CARDS:
    html += (
        f'<div style="width:130px;padding:10px;border:2px solid {color};border-radius:8px;text-align:center">'
        f'<div style="font-size:40px">{emoji}</div><b>{name}</b><br><span style="color:{color}">{typ}</span></div>'
    )
html += "</div>"
display(HTML(html))
print("Species cards rendered (offline).")


In [ ]:
import json
import os
import urllib.error
import urllib.request

NOTEBOOK_DEMO_VERSION = "v3-offline-default"
print(f"demo_cell_version={NOTEBOOK_DEMO_VERSION}")

# Default OFF in Docker — set POKEMON_USE_POKEAPI=1 to try live PokéAPI
USE_POKEAPI = os.environ.get("POKEMON_USE_POKEAPI", "0").strip() == "1"

FALLBACK_POKEMON = {
    "pikachu": {"name": "pikachu", "types": ["electric"], "hp": 35},
    "garchomp": {"name": "garchomp", "types": ["dragon", "ground"], "hp": 108},
    "charizard": {"name": "charizard", "types": ["fire", "flying"], "hp": 78},
    "blastoise": {"name": "blastoise", "types": ["water"], "hp": 79},
}


def pokeapi_get(path: str) -> dict:
    url = f"https://pokeapi.co/api/v2/{path.strip('/')}/"
    req = urllib.request.Request(url, headers={"User-Agent": "pokemon-rocm-tutorial/1.0"})
    with urllib.request.urlopen(req, timeout=8) as resp:
        return json.load(resp)


def pokemon_brief(name: str) -> dict:
    key = name.lower()
    if key not in FALLBACK_POKEMON:
        raise KeyError(f"unknown species {key}")
    if not USE_POKEAPI:
        return {**FALLBACK_POKEMON[key], "source": "builtin"}
    try:
        data = pokeapi_get(f"pokemon/{key}")
        types = [t["type"]["name"] for t in data["types"]]
        hp = next(s["base_stat"] for s in data["stats"] if s["stat"]["name"] == "hp")
        return {"name": data["name"], "types": types, "hp": hp, "source": "pokeapi"}
    except Exception as exc:
        print(f"[pokeapi] fallback for {key}: {type(exc).__name__}")
        return {**FALLBACK_POKEMON[key], "source": "fallback"}


TYPE_CHART = {
    ("electric", "water"): 2.0, ("electric", "ground"): 0.0,
    ("fire", "grass"): 2.0, ("water", "fire"): 2.0,
    ("ground", "electric"): 2.0, ("dragon", "dragon"): 2.0,
}


def type_multiplier(move_type: str, defender_types: list) -> float:
    m = 1.0
    for dt in defender_types:
        m *= TYPE_CHART.get((move_type, dt), 1.0)
    return m


class MiniShowdownDemo:
    def __init__(self, p2_names=("pikachu", "garchomp"), p1_names=("charizard", "blastoise")):
        self.p1 = [pokemon_brief(n) for n in p1_names]
        self.p2 = [pokemon_brief(n) for n in p2_names]
        self.p1_active, self.p2_active = 0, 0
        self.p1_hp = [p["hp"]] * len(self.p1)
        self.p2_hp = [p["hp"]] * len(self.p2)
        self.turn, self.log = 1, []

    def _active(self, side):
        mons = self.p1 if side == "p1" else self.p2
        idx = self.p1_active if side == "p1" else self.p2_active
        return mons[idx], idx

    def print_state(self):
        p1, _ = self._active("p1")
        p2, _ = self._active("p2")
        print(f"\n=== Turn {self.turn} ===")
        print(f"p1a: {p1['name']} HP={self.p1_hp[self.p1_active]} types={p1['types']} ({p1['source']})")
        print(f"p2a: {p2['name']} HP={self.p2_hp[self.p2_active]} types={p2['types']} ({p2['source']})")
        print("Bench p2:", [m["name"] for i, m in enumerate(self.p2) if i != self.p2_active])

    def apply_action(self, cmd: str) -> bool:
        cmd = cmd.strip().lower()
        if cmd.startswith("switch "):
            target = cmd.split(" ", 1)[1].strip().lower()
            for i, mon in enumerate(self.p2):
                if mon["name"].lower() == target and i != self.p2_active and self.p2_hp[i] > 0:
                    self.log.append(f"|switch|p2a: {mon['name'].title()}")
                    self.p2_active = i
                    return True
            print("Illegal switch — use a benched species name.")
            return False
        if cmd.startswith("move "):
            move_name = cmd.split(" ", 1)[1]
            p1, _ = self._active("p1")
            p2, _ = self._active("p2")
            mt = "electric" if "thunder" in move_name else "ground" if "earth" in move_name else p2["types"][0]
            dmg = max(1, int(20 * type_multiplier(mt, p1["types"])))
            self.p1_hp[self.p1_active] = max(0, self.p1_hp[self.p1_active] - dmg)
            self.log.append(f"|move|p2a: {p2['name'].title()}|{move_name.title()}|p1a: {p1['name'].title()}")
            print(f"Dealt ~{dmg} damage.")
            return True
        print("Use: move <name>  OR  switch <species>")
        return False

    def play_demo_round(self):
        self.print_state()
        for cmd in ("move thunderbolt", "switch garchomp", "move earthquake"):
            print(f"\n> p2: {cmd}")
            if not self.apply_action(cmd):
                raise RuntimeError(f"demo action failed: {cmd}")
            self.turn += 1
            self.print_state()
        print("\n--- protocol tail ---")
        for line in self.log[-4:]:
            print(line)


print("Mini Showdown demo (offline-safe)...")
MiniShowdownDemo().play_demo_round()
print("Step 3 demo OK")

try:
    import asyncio
    from poke_env.player import RandomPlayer
    from poke_env.ps_client.server_configuration import ShowdownServerConfiguration

    async def _ladder():
        p = RandomPlayer(server_configuration=ShowdownServerConfiguration, max_concurrent_battles=1)
        await p.ladder(1)
        await p.ps_client.stop_listening()

    print("\n[poke-env] optional ladder ...")
    asyncio.get_event_loop().run_until_complete(_ladder())
except Exception as exc:
    print(f"[poke-env] skipped ({exc})")


## Step 4 — Compare model weights (same demo prompt)
Load base `Qwen/Qwen3-4B` vs published `GoldenGrapeGentleman1/pokemon-showdown-agent-v6` one at a time.

In [ ]:
import gc
import os
import sys
from pathlib import Path

import torch

print(f"step4_cell_version=v2-smoke-then-compare")

_repo = Path.cwd().resolve()
_eval_root = None
for base in [_repo, *_repo.parents]:
    if (base / "scripts" / "eval" / "showdown_agent_eval.py").is_file():
        sys.path.insert(0, str(base / "scripts" / "eval"))
        _eval_root = base
        break
if _eval_root is None:
    raise FileNotFoundError("Open notebook from repo root containing scripts/eval/showdown_agent_eval.py")

from showdown_agent_eval import postprocess_agent_response, tutorial_demo_log_with_request

# --- Step 4a: smoke test (no model download) ---
USER = tutorial_demo_log_with_request()
assert "|request|" in USER or "|turn|" in USER
print(f"step4a_smoke_ok log_chars={len(USER)} eval_root={_eval_root}")
comparison_rows = []

# --- Step 4b: optional full model compare (GPU + HF) ---
RUN_MODEL_COMPARE = os.environ.get("POKEMON_RUN_MODEL_COMPARE", "1").strip() != "0"

if not torch.cuda.is_available():
    print("step4b_skipped: no GPU")
elif not RUN_MODEL_COMPARE:
    print("step4b_skipped: set POKEMON_RUN_MODEL_COMPARE=1 to enable")
else:
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    COMPARE_MODELS = [
        ("base", "Qwen/Qwen3-4B"),
        ("published_agent", "GoldenGrapeGentleman1/pokemon-showdown-agent-v6"),
    ]
    SYS = (
        "You are a Pokemon Showdown battle AI. You play as p2. "
        "Given the battle log, output your next action. "
        "Format: move <name> OR switch <name>. "
        "Append terastallize if you terastallize this turn."
    )

    def predict(m, tok, label):
        msgs = [{"role": "system", "content": SYS}, {"role": "user", "content": USER}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tok(text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = m.generate(**inp, max_new_tokens=48, temperature=0.1, do_sample=False, pad_token_id=tok.eos_token_id, use_cache=False)
        raw = tok.decode(out[0][inp.input_ids.shape[-1]:], skip_special_tokens=True)
        action = postprocess_agent_response(raw).splitlines()[0].strip()
        print(f"[{label}] {action}")
        return action

    for label, mid in COMPARE_MODELS:
        try:
            print(f"\nLoading {mid}...")
            m, tok = FastLanguageModel.from_pretrained(
                model_name=mid, max_seq_length=2048, dtype=torch.bfloat16,
                load_in_4bit=False, attn_implementation="sdpa",
            )
            tok = get_chat_template(tok, chat_template="qwen3")
            FastLanguageModel.for_inference(m)
            comparison_rows.append({"checkpoint": label, "model_id": mid, "action": predict(m, tok, label)})
            del m, tok
            gc.collect()
            torch.cuda.empty_cache()
        except Exception as exc:
            print(f"[{label}] FAILED: {exc}")
            comparison_rows.append({"checkpoint": label, "model_id": mid, "action": f"ERROR: {exc}"})

print("\n| checkpoint | action |")
if comparison_rows:
    for row in comparison_rows:
        print(f"| {row['checkpoint']} | `{row['action']}` |")
else:
    print("| (smoke only) | run with GPU + POKEMON_RUN_MODEL_COMPARE=1 |")
print("\nStep 4 OK — Step 6 reloads base Qwen for training.")


### Step 5a — Inspect one raw replay (long log)

In [ ]:

from datasets import load_dataset
row = next(iter(load_dataset("milkkarten/pokemon-showdown-replays-merged", split="train", streaming=True)))
log = row["log"]
print(f"chars={len(log)} lines={len(log.splitlines())}")
print("--- head ---\n" + "\n".join(log.splitlines()[:10]))
print("--- tail ---\n" + "\n".join(log.splitlines()[-6:]))
_raw_log = log


### Step 5b — Parse protocol + winner side

In [ ]:

def showdown_fields(line):
    p = line.strip().split("|")
    while p and p[0] == "": p = p[1:]
    return p

def extract_winner_side(log_text):
    winner, players = None, {}
    for line in log_text.split("\n"):
        f = showdown_fields(line)
        if len(f) >= 3 and f[0] == "player": players[f[1]] = f[2]
        if len(f) >= 2 and f[0] == "win": winner = f[1]
    if not winner: return None
    return next((s for s, n in players.items() if n == winner), None)

winner_side = extract_winner_side(_raw_log)
print(f"winner_side={winner_side}")


### Step 5c — Slice one turn (long → short prefix)

In [ ]:

MAX_LOG_CHARS = 6000
lines = _raw_log.strip().split("\n")
turns = [(int(f[1]), i) for i, ln in enumerate(lines) for f in [showdown_fields(ln)] if len(f) >= 2 and f[0] == "turn"]
idx = 0 if len(turns) == 2 else 1
_, tline = turns[idx]
nline = turns[idx + 1][1] if idx + 1 < len(turns) else len(lines)
action = None
for j in range(tline + 1, nline):
    f = showdown_fields(lines[j])
    if len(f) < 3: continue
    if f[0] == "move" and f[1].startswith(f"{winner_side}a:"): action = f"move {f[2]}"; break
    if f[0] == "switch" and f[1].startswith(f"{winner_side}a:"):
        action = f"switch {f[1].split(': ',1)[-1]}"; break
log_prefix = "\n".join(lines[: tline + 1])
print(f"action={action} prefix_len={len(log_prefix)} fits={len(log_prefix) <= MAX_LOG_CHARS}")
print("\n".join(log_prefix.splitlines()[-8:]))


### Step 5d — Chat row structure

In [ ]:

SYSTEM_TEMPLATE = "You are a Pokemon Showdown battle AI. You play as {side}. Given the battle log, output your next action. Format: move <name> OR switch <name>. Append terastallize if you terastallize this turn."
for role, body in [("system", SYSTEM_TEMPLATE.format(side=winner_side)), ("user", log_prefix[:120]+"..."), ("assistant", action)]:
    print(f"{role}: {body}")


## Step 6 — Load Qwen3-4B + LoRA
Attach adapters, then run Step 5e streaming to build `train_dataset`.


### Step 5e — Batch streaming (disk-safe)
Same logic as 5a–5d; **never** materializes the full corpus.

In [ ]:

from datasets import Dataset, load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

max_seq_length, dtype, load_in_4bit = 2048, torch.bfloat16, False
model, tokenizer = FastLanguageModel.from_pretrained(model_name="Qwen/Qwen3-4B", max_seq_length=max_seq_length, dtype=dtype, load_in_4bit=load_in_4bit, attn_implementation="sdpa")
tokenizer = get_chat_template(tokenizer, chat_template="qwen3")
model = FastLanguageModel.get_peft_model(model, r=64, target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], lora_alpha=128, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=3407)

MIN_RATING, MAX_TRAIN_SAMPLES, SHUFFLE_BUFFER, MAX_LOG_CHARS = 1400, 2000, 10_000, 6000

def format_sample(example):
    log_text, side = example["log"], extract_winner_side(example["log"])
    if not side: return {"text": ""}
    lines = log_text.strip().split("\n")
    turns = []
    for i, ln in enumerate(lines):
        f = showdown_fields(ln)
        if len(f) >= 2 and f[0] == "turn":
            try: turns.append((int(f[1]), i))
            except ValueError: pass
    if len(turns) < 2: return {"text": ""}
    ti = 0 if len(turns) == 2 else 1
    _, tline = turns[ti]
    nline = turns[ti+1][1] if ti+1 < len(turns) else len(lines)
    action = None
    for j in range(tline+1, nline):
        f = showdown_fields(lines[j])
        if len(f) < 3: continue
        if f[0] == "move" and f[1].startswith(f"{side}a:"):
            tera = ""
            if any("terastallize" in lines[k] and side in lines[k] for k in range(max(0,j-3), min(len(lines), j+3))):
                tera = " terastallize"
            action = f"move {f[2]}{tera}"; break
        if f[0] == "switch" and f[1].startswith(f"{side}a:"):
            action = f"switch {f[1].split(': ',1)[-1]}"; break
    if not action: return {"text": ""}
    prefix = "\n".join(lines[:tline+1])
    if len(prefix) > MAX_LOG_CHARS: return {"text": ""}
    msgs = [{"role":"system","content":SYSTEM_TEMPLATE.format(side=side)}, {"role":"user","content":prefix}, {"role":"assistant","content":action}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

stream = load_dataset("milkkarten/pokemon-showdown-replays-merged", split="train", streaming=True)
stream = stream.shuffle(seed=3407, buffer_size=SHUFFLE_BUFFER)
stream = stream.filter(lambda x: "gen9" in str(x.get("formatid") or "") and (x.get("rating") or 0) >= MIN_RATING)
train_samples, scanned = [], 0
for row in stream:
    scanned += 1
    f = format_sample(row)
    if f["text"]: train_samples.append(f)
    if len(train_samples) >= MAX_TRAIN_SAMPLES: break
train_dataset = Dataset.from_list(train_samples).shuffle(seed=3407)
print(f"examples={len(train_dataset)} scanned={scanned}")
print(train_dataset[0]["text"][:600])


## Step 7 — Quick validation SFT (50 steps)

In [ ]:

from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_dataset, dataset_text_field="text",
    max_seq_length=max_seq_length, packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=5, max_steps=50,
        learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, optim="adamw_torch", weight_decay=0.01, lr_scheduler_type="linear",
        seed=3407, output_dir="model_output_v6_demo", save_steps=50, save_total_limit=2, report_to="none",
    ),
)
trainer_stats = trainer.train()
print(trainer_stats)


## Step 8 — GRPO with designed reward function

| Signal | Weight | Meaning |
|--------|--------|---------|
| valid `move`/`switch` line | +2 / −3 | binary format gate |
| `move_legal_in_request` | +3 | matches `|request|` JSON |
| `switch_legal_in_request` | +3 | legal bench target |
| illegal tera | −2 | tera when not allowed |

Uses `showdown_agent_eval.validate_action_against_log` — same checks as inference.


In [ ]:

import importlib.util, sys
from pathlib import Path
from datasets import Dataset
from unsloth import is_bfloat16_supported

grpo_trainer = None
missing = [p for m,p in [("mergekit","mergekit"),("llm_blender","llm-blender")] if importlib.util.find_spec(m) is None]
if missing:
    print("Skip GRPO — pip install", " ".join(missing))
else:
    from trl import GRPOConfig, GRPOTrainer
    _repo = Path.cwd().resolve()
    for base in [_repo, *_repo.parents]:
        if (base / "scripts" / "eval" / "showdown_agent_eval.py").is_file():
            sys.path.insert(0, str(base / "scripts" / "eval")); break
    from showdown_agent_eval import validate_action_against_log

    def _log_from_prompt(prompt):
        m = "<|im_start|>user\n"
        if m in prompt:
            b = prompt.split(m, 1)[1]
            return b.split("<|im_start|>", 1)[0].strip() if "<|im_start|>" in b else b.strip()
        return ""

    def showdown_reward_func(prompts, completions, **kwargs):
        rewards = []
        for prompt, completion in zip(prompts, completions):
            text = completion[0]["content"] if isinstance(completion, list) else str(completion)
            line = text.strip().splitlines()[0] if text.strip() else ""
            log = _log_from_prompt(prompt if isinstance(prompt, str) else str(prompt))
            c = validate_action_against_log(line, log)
            r = 2.0 if c["structure_ok"] and c["single_line_ok"] else -3.0
            if c["request_present"]:
                if c.get("move_legal_in_request") is True: r += 3.0
                if c.get("switch_legal_in_request") is True: r += 3.0
                if c.get("tera_legal_in_request") is False: r -= 2.0
            rewards.append(r)
        return rewards

    grpo_ds = Dataset.from_list([
        {"prompt": s["text"].split("<|im_start|>assistant")[0] + "<|im_start|>assistant\n"}
        for s in train_samples[:64]
    ])
    grpo_trainer = GRPOTrainer(
        model=model, reward_funcs=[showdown_reward_func],
        args=GRPOConfig(output_dir="grpo_outputs", learning_rate=3e-6, per_device_train_batch_size=1,
            gradient_accumulation_steps=4, num_generations=4, max_completion_length=96, temperature=1.0,
            max_steps=10, logging_steps=1, report_to="none", fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported()),
        train_dataset=grpo_ds,
    )
    print("GRPO ready — uncomment train() below")
# if grpo_trainer: grpo_trainer.train()


## Step 9 — Inference sanity check

In [ ]:

import sys
from pathlib import Path
import torch

_repo = Path.cwd().resolve()
for base in [_repo, *_repo.parents]:
    if (base / "scripts" / "eval" / "showdown_agent_eval.py").is_file():
        sys.path.insert(0, str(base / "scripts" / "eval")); break
from showdown_agent_eval import postprocess_agent_response, tutorial_demo_log_with_request, validate_action_against_log

model.eval()
FastLanguageModel.for_inference(model)
user_msg = tutorial_demo_log_with_request()
msgs = [{"role":"system","content":"You are a Pokemon Showdown battle AI. You play as p2. Given the battle log, output your next action. Format: move <name> OR switch <name>. Append terastallize if you terastallize this turn."}, {"role":"user","content":user_msg}]
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inp = tokenizer(text, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(**inp, max_new_tokens=64, temperature=0.1, do_sample=False, pad_token_id=tokenizer.eos_token_id, use_cache=False)
raw = tokenizer.decode(out[0][inp.input_ids.shape[-1]:], skip_special_tokens=True)
resp = postprocess_agent_response(raw)
line = resp.splitlines()[0].strip() if resp.strip() else ""
checks = validate_action_against_log(line, user_msg)
print("--- prediction ---\n", resp)
print("--- checks ---", {k: checks[k] for k in ["structure_ok","single_line_ok","request_present","move_legal_in_request","switch_legal_in_request"]})
print("\nStep 4 baselines:")
for r in globals().get("comparison_rows", []):
    print(f"  {r['checkpoint']}: {r['action']}")


## Step 10 — Export to Hugging Face
```python
export_dir = "model_output_v6_demo/merged_model"
model.save_pretrained_merged(export_dir, tokenizer, save_method="merged_16bit")
```